In [1]:
workspace_id = "2f59ccc2-e9d6-44b5-8c0c-1bdbf1fe148a"
bronze_lakehouse_id = "33b93b7e-0899-43c6-81a8-654bc740aa32"
url_gdp = "https://api.worldbank.org/v2/country/USA/indicator/NY.GDP.MKTP.CD?format=json&per_page=1000"

StatementMeta(, d30f389a-e672-44fd-929f-6b2c17383443, 3, Finished, Available, Finished, False)

In [2]:
import json
import requests

bronze_gdp_path = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{bronze_lakehouse_id}/Tables/dbo/gdp_bronze"

response = requests.get(url_gdp, timeout=30)
response.raise_for_status()

data = response.json()

if len(data) > 1 and data[1]:
    results = data[1]
    json_rdd = spark.sparkContext.parallelize(
        [json.dumps(r, ensure_ascii=False) for r in results]
    )
    df_gdp = spark.read.json(json_rdd)
    df_gdp.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(bronze_gdp_path)
    print("GDP data successfully loaded into gdp_bronze.")
    display(df_gdp.limit(5))
else:
    print("No data returned from the World Bank API.")

StatementMeta(, d30f389a-e672-44fd-929f-6b2c17383443, 4, Finished, Available, Finished, False)

GDP data successfully loaded into gdp_bronze.


SynapseWidget(Synapse.DataFrame, ce38f566-39df-4dcd-ac72-2cdc1e38eb6e)